# Automatic evaluation of the NLLB conditions

Scores the translations of the four conditions with two metric sets, and tests which differences between conditions are significant.

- **§1** follows Vico & Libovický (2026): BLEU, chrF++, TER, COMET
- **§2** follows Casola et al. (2025): ROUGE-L, METEOR, BERTScore, BLEURT
- **§3** paired bootstrap tests for every pair of conditions

The input is `it-pms_test_translations.csv` from [offelia39/pms-wiki-mined-corpora](https://huggingface.co/datasets/offelia39/pms-wiki-mined-corpora), the output of the fine-tuning notebook plus a column of references respelled in standard orthography. `REF_COL` in §0 picks which reference column is scored against. Results go to `OUT_DIR` on Drive, one folder per reference column.

### Running it

COMET (§1) and BLEURT (§2) need conflicting package versions, so they have to run on separate runtimes:

1. **§1**: run the install cell, restart the session (*Runtime → Restart session*), then §0 and the rest of §1.
2. **§2**: start from a new runtime (*Runtime → Disconnect and delete runtime*; a restart isn't enough, installed packages survive it). Run the install cell, restart, then §0 and the rest of §2.
3. **§3**: any runtime, e.g. straight after §2. Run §0 first. It reads the per-sentence scores that §1 and §2 saved.

A GPU runtime is needed for §1 and §2. To score against the other reference column, change `REF_COL` and go through the three steps again.

## §0 Setup

Run first in every session, and again after every restart. `HF_HOME` keeps the downloaded metric models on Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = "/content/drive/MyDrive/Tesi_piedmontese/"
os.environ["HF_HOME"] = BASE + "hf_cache"   # before importing evaluate / comet

REF_COL = "reference"                 # speakers' original spelling
# REF_COL = "reference_std_revised"   # standard orthography

CONDITIONS = ["NLLB-avg", "NLLB-avg+MT", "NLLB-avg+DAE-50K+MT", "NLLB-avg+DAE-full+MT"]
COLORS     = ["#F2A7A7", "#E5636E", "#C8102E", "#7A0A1C"]   # light to dark red, one per condition

OUT_DIR = BASE + f"metrics/{REF_COL}/"
os.makedirs(OUT_DIR, exist_ok=True)

# while the dataset is private, log in first:
# from huggingface_hub import login; login()
from huggingface_hub import hf_hub_download
import pandas as pd

csv_path = hf_hub_download("offelia39/pms-wiki-mined-corpora", "it-pms_test_translations.csv",
                           repo_type="dataset")
df = pd.read_csv(csv_path, keep_default_na=False)   # an empty translation stays "", not NaN
print(f"{len(df)} sentences, scored against {REF_COL!r}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def bar_chart(scores, metrics, labels, title, ylabel, legend_loc, png):
    """Grouped bars: one group per metric, one bar per condition. scores[cond][metric]."""
    bw = 0.18
    x = np.arange(len(metrics))
    off = (len(CONDITIONS) - 1) * bw / 2
    fig, ax = plt.subplots(figsize=(11, 6))
    for i, cond in enumerate(CONDITIONS):
        ax.bar(x + i * bw - off, [scores[cond][m] for m in metrics], bw,
               label=cond, color=COLORS[i], edgecolor="white", linewidth=0.5)
    ax.set_xticks(x)
    ax.set_xticklabels([labels[m] for m in metrics], fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(title, fontsize=13)
    ax.legend(fontsize=10, loc=legend_loc)
    ax.grid(axis="y", alpha=0.35, linestyle="--")
    ax.set_ylim(bottom=0)
    plt.tight_layout()
    plt.savefig(png, dpi=150, bbox_inches="tight")
    plt.show()

## §1 Vico & Libovický (2026) metrics
out: `vl_metrics.csv`, `vl_comet_per_sentence.csv`, `table1_vl.csv`, `vl_metrics.png`

- BLEU: lowercased
- chrF++: lowercased, word n-grams up to 2
- TER: case-insensitive, punctuation removed, normalised; lower is better
- COMET: `wmt22-comet-da`

The value in parentheses is a bootstrap standard error (1000 resamples): the width of the 95% percentile interval divided by 2 × 1.96. For BLEU, chrF++ and TER the whole corpus score is recomputed on each resample; for COMET the sentence scores are resampled (`scipy.stats.bootstrap`).

In [ ]:
# Session 1 only. Afterwards: Runtime → Restart session, then §0 and the next cell.
!pip install -q "unbabel-comet>=2.2.2" sacrebleu
!pip install -q "protobuf>=5.28"
!pip install -q "numpy>=2,<2.3"   # back to numpy 2; <2.3 for numba

In [ ]:
# COMET was written for older transformers; three patches make it load and predict
import collections, dataclasses
import transformers.utils
from transformers.utils import generic as _generic

# 1) a function removed in transformers 4.40
if not hasattr(transformers.utils, "is_torch_tpu_available"):
    transformers.utils.is_torch_tpu_available = lambda full_output=False: False

# 2) ModelOutput insists on dataclass subclasses
_orig_init = _generic.ModelOutput.__init__
def _permissive_init(self, *args, **kwargs):
    if dataclasses.is_dataclass(type(self)):
        _orig_init(self, *args, **kwargs)
    else:
        collections.OrderedDict.__init__(self)
        for k, v in kwargs.items():
            self[k] = v
_generic.ModelOutput.__init__ = _permissive_init

from comet import download_model, load_from_checkpoint
import comet.models.utils as _cmu
import comet.models.base as _cmb

# 3) COMET's Prediction class (a ModelOutput) replaced by a plain dict-like one
class _Prediction:
    def __init__(self, **kwargs): self.__dict__.update(kwargs)
    def __iter__(self): return iter(self.__dict__)
    def __contains__(self, k): return k in self.__dict__
    def __getitem__(self, k): return self.__dict__[k]
    def __setitem__(self, k, v): self.__dict__[k] = v
    def keys(self): return self.__dict__.keys()
_cmu.Prediction = _cmb.Prediction = _Prediction

import torch
gpus = 1 if torch.cuda.is_available() else 0
comet_model = load_from_checkpoint(download_model("Unbabel/wmt22-comet-da"))   # ~2.3 GB

In [ ]:
import sacrebleu
from scipy.stats import bootstrap

VL_METRICS = {
    "BLEU":   sacrebleu.BLEU(lowercase=True),
    "chrF++": sacrebleu.CHRF(lowercase=True, word_order=2),
    "TER":    sacrebleu.TER(case_sensitive=False, no_punct=True, normalized=True),
}

def corpus_se(metric, hyps, refs, n=1000, seed=42):
    rng = np.random.default_rng(seed)
    scores = []
    for _ in range(n):
        idx = rng.integers(0, len(hyps), size=len(hyps))
        scores.append(metric.corpus_score([hyps[i] for i in idx], [[refs[i] for i in idx]]).score)
    lo, hi = np.percentile(scores, [2.5, 97.5])
    return (hi - lo) / (2 * 1.96)

srcs = df["source"].tolist()
refs = df[REF_COL].tolist()
vl_rows, comet_sent = [], {}

for cond in CONDITIONS:
    hyps = df[cond].tolist()
    row = {"Model": cond}
    for name, metric in VL_METRICS.items():
        row[name]         = round(metric.corpus_score(hyps, [refs]).score, 2)
        row[name + "_SE"] = round(corpus_se(metric, hyps, refs), 2)

    out = comet_model.predict([{"src": s, "mt": h, "ref": r} for s, h, r in zip(srcs, hyps, refs)],
                              batch_size=16, gpus=gpus)
    sent = np.array(out.scores)
    ci = bootstrap((sent,), np.mean, vectorized=True, rng=42, n_resamples=1000).confidence_interval
    lo, hi = round(ci.low, 4), round(ci.high, 4)
    row["COMET"]    = round(out.system_score, 4)
    row["COMET_SE"] = round((hi - lo) / (2 * 1.96), 4)
    row["COMET_CI"] = f"[{lo:.4f}, {hi:.4f}]"

    vl_rows.append(row)
    comet_sent[cond] = sent
    print(cond, {k: row[k] for k in ["BLEU", "chrF++", "TER", "COMET"]})

vl_df = pd.DataFrame(vl_rows)
vl_df.to_csv(OUT_DIR + "vl_metrics.csv", index=False)
pd.DataFrame({"flores_id": df["flores_id"], **comet_sent}).to_csv(
    OUT_DIR + "vl_comet_per_sentence.csv", index=False)

In [ ]:
def with_se(v, s, dp):
    return f"{v:.{dp}f} ({s:.{dp}f})"

table1 = pd.DataFrame({
    "Model":  vl_df["Model"],
    "BLEU":   [with_se(v, s, 2) for v, s in zip(vl_df["BLEU"],   vl_df["BLEU_SE"])],
    "chrF++": [with_se(v, s, 2) for v, s in zip(vl_df["chrF++"], vl_df["chrF++_SE"])],
    "TER ↓":  [with_se(v, s, 2) for v, s in zip(vl_df["TER"],    vl_df["TER_SE"])],
    "COMET":  [with_se(v, s, 4) for v, s in zip(vl_df["COMET"],  vl_df["COMET_SE"])],
})
table1.to_csv(OUT_DIR + "table1_vl.csv", index=False)
print(table1.to_string(index=False))

vl_scores = {r["Model"]: {**r, "COMET": r["COMET"] * 100} for r in vl_rows}   # COMET ×100 for the plot
bar_chart(vl_scores, ["BLEU", "chrF++", "TER", "COMET"],
          {"BLEU": "BLEU", "chrF++": "chrF++", "TER": "TER ↓", "COMET": "COMET (×100)"},
          "Vico & Libovický (2026) metrics", "Score", "upper left", OUT_DIR + "vl_metrics.png")

## §2 Casola et al. (2025) metrics
out: `casola_metrics.csv`, `casola_per_sentence.csv`, `table2_casola.csv`, `casola_metrics.png`

*Start from a new runtime* (see the top of the notebook).

- ROUGE-1, -2, -L: F1 per sentence with Porter stemming, averaged
- METEOR: `evaluate` with NLTK defaults
- BERTScore: F1, `roberta-large`, layer 17, no IDF
- BLEURT: BLEURT-20
- BLEU: sacrebleu defaults (case-sensitive); only in the plot

All scores ×100. In Table 2 the value in parentheses is the standard deviation of the sentence scores.

In [ ]:
# Session 2 only, on a new runtime. Afterwards: Runtime → Restart session, then §0 and the next cell.
!pip install -q evaluate bert-score sacrebleu
!pip install -q "huggingface-hub<2.0"
!pip install -q git+https://github.com/google-research/bleurt.git

In [ ]:
import re
from collections import Counter

import evaluate
import nltk
from bert_score import score as bert_score
from nltk.stem import PorterStemmer
from sacrebleu.metrics import BLEU

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

_stemmer = PorterStemmer()

def _tokens(text):
    return [_stemmer.stem(t) for t in re.sub(r"[^a-z0-9\s]", " ", text.lower()).split()]

def _f1(overlap, n_hyp, n_ref):
    if overlap == 0 or not n_hyp or not n_ref:
        return 0.0
    p, r = overlap / n_hyp, overlap / n_ref
    return 2 * p * r / (p + r)

def _ngrams(tokens, n):
    return Counter(zip(*[tokens[i:] for i in range(n)]))

def _lcs(x, y):
    prev = [0] * (len(y) + 1)
    for xi in x:
        curr = [0] * (len(y) + 1)
        for j, yj in enumerate(y):
            curr[j + 1] = prev[j] + 1 if xi == yj else max(curr[j], prev[j + 1])
        prev = curr
    return prev[-1]

def rouge_n_sent(hyps, refs, n):
    out = []
    for h, r in zip(hyps, refs):
        hn, rn = _ngrams(_tokens(h), n), _ngrams(_tokens(r), n)
        out.append(_f1(sum((hn & rn).values()), sum(hn.values()), sum(rn.values())) * 100)
    return out

def rouge_l_sent(hyps, refs):
    out = []
    for h, r in zip(hyps, refs):
        ht, rt = _tokens(h), _tokens(r)
        out.append(_f1(_lcs(ht, rt), len(ht), len(rt)) * 100)
    return out

_meteor = evaluate.load("meteor")
def meteor_sent(hyps, refs):
    return [_meteor.compute(predictions=[h], references=[r])["meteor"] * 100 for h, r in zip(hyps, refs)]

def bertscore_sent(hyps, refs):
    _, _, f1 = bert_score(hyps, refs, model_type="roberta-large", num_layers=17, verbose=False)
    return [s * 100 for s in f1.tolist()]

try:
    _bleurt = evaluate.load("bleurt", "BLEURT-20")
except Exception as e:
    _bleurt = None
    print(f"BLEURT not available, skipped ({e})")

def bleurt_sent(hyps, refs):
    return [s * 100 for s in _bleurt.compute(predictions=hyps, references=refs)["scores"]]

_bleu = BLEU()

In [ ]:
refs = df[REF_COL].astype(str).tolist()
cp_scores, per_sent = {}, []

for cond in CONDITIONS:
    hyps = df[cond].astype(str).tolist()
    sent = {"R-L":       rouge_l_sent(hyps, refs),
            "METEOR":    meteor_sent(hyps, refs),
            "BERTScore": bertscore_sent(hyps, refs)}
    if _bleurt is not None:
        sent["BLEURT"] = bleurt_sent(hyps, refs)

    row = {"R-1":  np.mean(rouge_n_sent(hyps, refs, 1)),
           "R-2":  np.mean(rouge_n_sent(hyps, refs, 2)),
           "BLEU": _bleu.corpus_score(hyps, [refs]).score}
    for m, s in sent.items():
        row[m] = np.mean(s)
        row[m + "_SD"] = np.std(s, ddof=1)
    cp_scores[cond] = row
    per_sent.append(pd.DataFrame({"flores_id": df["flores_id"], "Model": cond, **sent}))
    print(cond, "done")

cp_df = pd.DataFrame([{"Model": c, **cp_scores[c]} for c in CONDITIONS]).round(2)
cp_df.to_csv(OUT_DIR + "casola_metrics.csv", index=False)
pd.concat(per_sent).to_csv(OUT_DIR + "casola_per_sentence.csv", index=False)

In [ ]:
TABLE2 = {"R-L": "ROUGE-L", "METEOR": "METEOR", "BERTScore": "BERTScore", "BLEURT": "BLEURT"}
table2 = pd.DataFrame({"Model": CONDITIONS, **{
    label: [f"{cp_scores[c][m]:.2f} ({cp_scores[c][m + '_SD']:.2f})" if m in cp_scores[c] else "n/a"
            for c in CONDITIONS]
    for m, label in TABLE2.items()}})
table2.to_csv(OUT_DIR + "table2_casola.csv", index=False)
print(table2.to_string(index=False))

plot_metrics = [m for m in ["R-L", "BLEU", "METEOR", "BERTScore", "BLEURT"] if m in cp_scores[CONDITIONS[0]]]
bar_chart(cp_scores, plot_metrics,
          {"R-L": "ROUGE-L", "BLEU": "BLEU", "METEOR": "METEOR", "BERTScore": "BERTScore", "BLEURT": "BLEURT"},
          "Casola et al. (2025) metrics", "Score (×100)", "lower right", OUT_DIR + "casola_metrics.png")

## §3 Significance
in: `vl_comet_per_sentence.csv`, `casola_per_sentence.csv` · out: `significance.csv`

Paired bootstrap resampling (Koehn 2004), two-sided and centred as in `sacrebleu --paired-bs`, with 1000 resamples shared by all conditions and metrics. BLEU, chrF++ and TER are recomputed on each resample with the §1 settings; the other metrics are averages of the sentence scores saved in §1 and §2. Each line is the difference B − A with its 95% interval. p-values are Holm-corrected within each metric, over the six pairs. For TER, a negative difference is an improvement.

In [ ]:
!pip install -q sacrebleu
import itertools
import sacrebleu

N_BOOT, SEED = 1000, 12345
PAIRS = list(itertools.combinations(CONDITIONS, 2))

refs = df[REF_COL].tolist()
hyps = {c: df[c].tolist() for c in CONDITIONS}

comet = pd.read_csv(OUT_DIR + "vl_comet_per_sentence.csv")
casola = pd.read_csv(OUT_DIR + "casola_per_sentence.csv")
assert comet["flores_id"].tolist() == df["flores_id"].tolist()
sent = {}
for c in CONDITIONS:
    sub = casola[casola["Model"] == c]
    assert sub["flores_id"].tolist() == df["flores_id"].tolist()
    sent[c] = {"COMET": comet[c].to_numpy(),
               **{m: sub[m].to_numpy() for m in ["R-L", "METEOR", "BERTScore", "BLEURT"] if m in sub}}

CORPUS = {"BLEU":   sacrebleu.BLEU(lowercase=True),
          "chrF++": sacrebleu.CHRF(lowercase=True, word_order=2),
          "TER":    sacrebleu.TER(case_sensitive=False, no_punct=True, normalized=True)}
METRICS = list(CORPUS) + list(sent[CONDITIONS[0]])

N = len(refs)
rng = np.random.default_rng(SEED)
samples = [np.arange(N)] + [rng.integers(0, N, size=N) for _ in range(N_BOOT)]   # [0] = full set

def score(c, m, idx):
    if m in CORPUS:
        return CORPUS[m].corpus_score([hyps[c][i] for i in idx], [[refs[i] for i in idx]]).score
    return float(sent[c][m][idx].mean())

boot = {(c, m): np.array([score(c, m, idx) for idx in samples]) for c in CONDITIONS for m in METRICS}

def holm(p):
    p = np.asarray(p); adj = np.empty_like(p); running = 0.0
    for k, i in enumerate(np.argsort(p)):
        running = max(running, min(1.0, (len(p) - k) * p[i]))
        adj[i] = running
    return adj

rows = []
for m in METRICS:
    block = []
    for a, b in PAIRS:
        d = boot[(b, m)] - boot[(a, m)]
        obs, d_boot = d[0], d[1:]
        p = (np.sum(np.abs(d_boot - d_boot.mean()) >= abs(obs)) + 1) / (N_BOOT + 1)
        lo, hi = np.percentile(d_boot, [2.5, 97.5])
        block.append({"Metric": m, "A": a, "B": b, "Delta (B−A)": round(obs, 4),
                      "95% CI": f"[{lo:.3f}, {hi:.3f}]", "p": p})
    for r, p_adj in zip(block, holm([r["p"] for r in block])):
        r["p_holm"] = p_adj
        r["sig"] = "*" if p_adj < 0.05 else ""
    rows += block

sig_df = pd.DataFrame(rows).round({"p": 4, "p_holm": 4})
sig_df.to_csv(OUT_DIR + "significance.csv", index=False)
print(sig_df.to_string(index=False))